# 🎃 ZapalloAI — Notebook 02: Preprocesamiento y Unificación

**Universidad de las Fuerzas Armadas ESPE**

Este notebook:
1. Unifica las clases de los dos datasets con el mapeo real de carpetas
2. Elimina duplicados entre datasets
3. Realiza un split estratificado **70% train / 15% val / 15% test**
4. Aplica augmentation offline para clases minoritarias
5. Genera estadísticas y el `dataset_final.yaml`

## Estructura real detectada

### Cucurbit Leaf Disease Dataset
```
raw/Cucurbit_leaf/
├── Downy mildew/       → downy_mildew   (1000 imgs)
├── Healthy/            → healthy         (711 imgs)
├── Leaf curl disease/  → leaf_curl       (1400 imgs)
└── Mosaic virus/       → mosaic_virus    (1010 imgs)
```

### Sweet Pumpkin Disease Recognition
```
raw/sweet_pumpkin/
├── Augmented Images/
│   ├── Augmented Sweet Pumpkin Downy Mildew Disease/ → downy_mildew  (1400)
│   ├── Augmented Sweet Pumpkin Healthy Leaf/          → healthy       (1400)
│   ├── Augmented Sweet Pumpkin Leaf Curl Disease/     → leaf_curl     (1400)
│   ├── Augmented Sweet Pumpkin Mosaic Disease/        → mosaic_virus  (1400)
│   └── Augmented Sweet Pumpkin Red Beetle/            → red_beetle    (1400)
└── Original Images/   (200 por clase — subconjunto de Augmented, omitir)
```

**Total disponible: ~11,121 imágenes en 5 clases**

In [ ]:
# ── 0. Instalar dependencias ──────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'albumentations', 'imagehash', 'opencv-python-headless',
    'Pillow', 'matplotlib', 'pandas'
], check=False)
print('✅ Dependencias listas')

In [ ]:
# ── 1. Configuración de rutas ─────────────────────────────────────
import os, sys, shutil, random
import numpy as np
from pathlib import Path
from PIL import Image
import imagehash

# ─── CAMBIAR AQUÍ SEGÚN DÓNDE EJECUTAS ───────────────────────────
COLAB_MODE = False   # False = Local | True = Google Colab + Drive
# ─────────────────────────────────────────────────────────────────

if COLAB_MODE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_RAW = Path('/content/drive/MyDrive/ZapalloAI')
else:
    # Detectar raíz del repositorio automáticamente
    ROOT = Path(os.path.abspath('')).resolve()
    for _ in range(6):
        if (ROOT / 'model').exists() and (ROOT / 'zapallo_app').exists():
            break
        ROOT = ROOT.parent
    BASE_RAW = ROOT / 'model' / 'data' / 'raw'

# Rutas de cada dataset (con los nombres reales de las carpetas)
CUCURBIT_DIR   = BASE_RAW / 'Cucurbit_leaf'
SWEET_AUG_DIR  = BASE_RAW / 'sweet_pumpkin' / 'Augmented Images'
# Sweet Original (200 imgs) es subconjunto de Augmented — se omite para evitar duplicados

OUTPUT_DIR = BASE_RAW.parent / 'processed'

# 5 clases objetivo
CLASSES = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

# ─── Mapeo Cucurbit Leaf (nombre carpeta → clase normalizada) ─────
CUCURBIT_MAP = {
    'Downy mildew':      'downy_mildew',
    'Healthy':           'healthy',
    'Leaf curl disease': 'leaf_curl',
    'Mosaic virus':      'mosaic_virus',
}

# ─── Mapeo Sweet Pumpkin Augmented ────────────────────────────────
SWEET_MAP = {
    'Augmented Sweet Pumpkin Downy Mildew Disease': 'downy_mildew',
    'Augmented Sweet Pumpkin Healthy Leaf':         'healthy',
    'Augmented Sweet Pumpkin Leaf Curl Disease':    'leaf_curl',
    'Augmented Sweet Pumpkin Mosaic Disease':       'mosaic_virus',
    'Augmented Sweet Pumpkin Red Beetle':           'red_beetle',
}

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'Modo           : {"Google Colab" if COLAB_MODE else "Local"}')
print(f'Cucurbit dir   : {CUCURBIT_DIR}')
print(f'Sweet Aug dir  : {SWEET_AUG_DIR}')
print(f'Output dir     : {OUTPUT_DIR}')

# Verificar existencia
for d, label in [(CUCURBIT_DIR, 'Cucurbit'), (SWEET_AUG_DIR, 'Sweet Pumpkin Augmented')]:
    if d.exists():
        clases = [x.name for x in sorted(d.iterdir()) if x.is_dir()]
        print(f'  ✅ {label}: {len(clases)} clases encontradas')
        for c in clases:
            n = len(list((d/c).glob('*.jpg'))) + len(list((d/c).glob('*.png')))
            print(f'       {c}: {n} imgs')
    else:
        print(f'  ❌ NO encontrado: {d}')

In [ ]:
# ── 2. Recopilar todas las imágenes y detectar duplicados ─────────
def get_images(folder: Path) -> list:
    """Retorna todas las imágenes JPG/PNG de una carpeta."""
    if not folder.exists():
        return []
    result = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        result.extend(folder.glob(ext))
    return result

# Recopilar todas las imágenes de ambos datasets
all_imgs_by_cls = {}  # clase_normalizada → [paths]

# — Cucurbit Leaf —
for folder_name, cls_name in CUCURBIT_MAP.items():
    imgs = get_images(CUCURBIT_DIR / folder_name)
    all_imgs_by_cls.setdefault(cls_name, []).extend(imgs)
    print(f'  Cucurbit {folder_name}: {len(imgs)} imgs → {cls_name}')

# — Sweet Pumpkin Augmented —
for folder_name, cls_name in SWEET_MAP.items():
    imgs = get_images(SWEET_AUG_DIR / folder_name)
    all_imgs_by_cls.setdefault(cls_name, []).extend(imgs)
    print(f'  Sweet {folder_name[:40]}: {len(imgs)} imgs → {cls_name}')

print('\n📊 Total por clase (antes de deduplicación):')
grand = 0
for cls in CLASSES:
    n = len(all_imgs_by_cls.get(cls, []))
    grand += n
    print(f'  {cls:<20}: {n:>5} imgs')
print(f'  {"TOTAL":<20}: {grand:>5} imgs')

In [ ]:
# ── 3. Deduplicación con pHash (dentro de cada clase) ─────────────
# Nota: Sweet Pumpkin ya está aumentado, así que los "duplicados" pueden ser
# versiones augmentadas de la misma imagen. Se eliminan solo hashes idénticos.

deduped_by_cls = {}  # clase → [paths únicos]

for cls in CLASSES:
    imgs = all_imgs_by_cls.get(cls, [])
    seen_hashes = {}
    unique = []
    for img_path in imgs:
        try:
            with Image.open(img_path) as img:
                h = str(imagehash.phash(img))
            if h not in seen_hashes:
                seen_hashes[h] = str(img_path)
                unique.append(img_path)
        except Exception:
            pass  # Ignorar imágenes corruptas
    
    removed = len(imgs) - len(unique)
    deduped_by_cls[cls] = unique
    print(f'  {cls:<20}: {len(imgs):>5} → {len(unique):>5} únicas (−{removed} dups)')

total_unique = sum(len(v) for v in deduped_by_cls.values())
print(f'\n✅ Total único: {total_unique:,} imágenes')

In [ ]:
# ── 4. Crear estructura de salida y split estratificado ───────────
# Crear carpetas
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print(f'✅ Estructura creada: {OUTPUT_DIR}')

# Split y copia
split_stats = {cls: {'train': 0, 'val': 0, 'test': 0} for cls in CLASSES}

for cls in CLASSES:
    imgs = deduped_by_cls.get(cls, []).copy()
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        'train': imgs[:n_train],
        'val':   imgs[n_train:n_train + n_val],
        'test':  imgs[n_train + n_val:],
    }

    for split_name, split_imgs in splits.items():
        for idx, img_path in enumerate(split_imgs):
            # Prefijo para evitar colisiones entre datasets
            src = 'swe' if 'sweet_pumpkin' in str(img_path) else 'cuc'
            dst_name = f'{src}_{img_path.name}'
            dst_path = OUTPUT_DIR / split_name / cls / dst_name
            if not dst_path.exists():
                shutil.copy2(img_path, dst_path)
            split_stats[cls][split_name] += 1

    print(f'  ✅ {cls:<20}: '
          f'train={split_stats[cls]["train"]:>5}, '
          f'val={split_stats[cls]["val"]:>4}, '
          f'test={split_stats[cls]["test"]:>4}')

print('\n✅ Split completado')

In [ ]:
# ── 5. Tabla resumen y gráfico de distribución ────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans'})

print(f"{'Clase':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'TOTAL':>8}")
print('─' * 52)
grand_total = 0
for cls in CLASSES:
    t  = split_stats[cls]['train']
    v  = split_stats[cls]['val']
    ts = split_stats[cls]['test']
    tot = t + v + ts
    grand_total += tot
    print(f"{cls:<20} {t:>8} {v:>8} {ts:>8} {tot:>8}")
print('─' * 52)
tt = sum(split_stats[c]['train'] for c in CLASSES)
tv = sum(split_stats[c]['val']   for c in CLASSES)
ts = sum(split_stats[c]['test']  for c in CLASSES)
print(f"{'TOTAL':<20} {tt:>8} {tv:>8} {ts:>8} {grand_total:>8}")

# Balance de clases
train_ns = [split_stats[c]['train'] for c in CLASSES]
ratio = max(train_ns) / min(train_ns) if min(train_ns) > 0 else 0
print(f'\nBalance max/min (train): {ratio:.2f}x', '✅' if ratio <= 3 else '⚠️ Desbalanceado')

# Gráfico
x = range(len(CLASSES))
t_v = [split_stats[c]['train'] for c in CLASSES]
v_v = [split_stats[c]['val']   for c in CLASSES]
s_v = [split_stats[c]['test']  for c in CLASSES]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, t_v, label='Train', color='#2D6A4F')
ax.bar(x, v_v, bottom=t_v, label='Val', color='#52B788')
ax.bar(x, s_v, bottom=[a+b for a,b in zip(t_v,v_v)], label='Test', color='#95D5B2')
ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.set_ylabel('Número de imágenes')
ax.set_title('ZapalloAI — Distribución del Dataset Procesado')
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR.parent / 'dataset_distribution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico: {fig_path}')

In [ ]:
# ── 6. Augmentation para clases minoritarias ──────────────────────
import albumentations as A
import cv2

pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=40, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=20, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(10, 30),
                    hole_width_range=(10, 30), p=0.3),
])

TARGET = max(train_ns)  # Igualar a la clase más grande
print(f'Objetivo por clase en train: {TARGET} imgs')

aug_total = 0
for cls in CLASSES:
    train_dir = OUTPUT_DIR / 'train' / cls
    existing  = list(train_dir.glob('*.jpg')) + list(train_dir.glob('*.png'))
    current   = len(existing)
    needed    = max(0, TARGET - current)

    if needed == 0:
        print(f'  OK  {cls:<20}: {current} imgs')
        continue

    print(f'  AUG {cls:<20}: {current} → +{needed}...')
    gen = 0
    while gen < needed:
        src_path = random.choice(existing)
        bgr = cv2.imread(str(src_path))
        if bgr is None:
            continue
        rgb     = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        aug_rgb = pipeline(image=rgb)['image']
        aug_bgr = cv2.cvtColor(aug_rgb, cv2.COLOR_RGB2BGR)
        out = train_dir / f'aug_{cls}_{gen:05d}.jpg'
        cv2.imwrite(str(out), aug_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
        gen += 1
    aug_total += gen

print(f'\n✅ Augmentation offline: +{aug_total} imgs generadas')

In [ ]:
# ── 7. Guardar dataset_final.yaml ────────────────────────────────
yaml_path = OUTPUT_DIR.parent / 'dataset_final.yaml'

yaml_content = f"""# Dataset ZapalloAI — generado por Notebook 02
# Fuentes: Cucurbit Leaf Disease Dataset + Sweet Pumpkin Disease Recognition

path: {OUTPUT_DIR.resolve().as_posix()}
train: train
val: val
test: test

nc: 5
names:
  0: healthy
  1: downy_mildew
  2: leaf_curl
  3: mosaic_virus
  4: red_beetle
"""

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

# Contar total final
total_final = sum(
    sum(1 for _ in (OUTPUT_DIR / split / cls).glob('*.*'))
    for split in ['train', 'val', 'test']
    for cls in CLASSES
    if (OUTPUT_DIR / split / cls).exists()
)

print(f'✅ dataset_final.yaml → {yaml_path}')
print(f'📊 Total imágenes procesadas: {total_final:,}')
print(f'\n🚀 SIGUIENTE: ejecutar Notebook 03 (entrenamiento YOLOv11n-cls)')
print(f'   data = "{OUTPUT_DIR.resolve().as_posix()}"')